# Notebook 1: Introducción a AWS Bedrock — Invocación de Modelos Base

En este notebook aprenderás a:
- Conectarte a AWS Bedrock usando `boto3`
- Listar los modelos disponibles en tu región
- Invocar un modelo de texto (Claude) con la API `invoke_model`
- Entender la estructura de requests y responses

---

## Prerrequisitos

- Cuenta de AWS con acceso a Bedrock habilitado
- Credenciales configuradas (`aws configure` o variables de entorno)
- Modelos activados en la consola de AWS Bedrock (Model Access)

```bash
pip install boto3
```

## 1. Instalación de dependencias

In [ ]:
# !pip install boto3 --quiet

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
AWS_BEARER_TOKEN_BEDROCK=os.getenv("AWS_BEARER_TOKEN_BEDROCK")
AWS_ACCESS_KEY_ID=os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY=os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_DEFAULT_REGION=os.getenv("AWS_DEFAULT_REGION")

# API key de Gemini
# API_KEY = userdata.get('GEMINI_API_KEY')

## 2. Importaciones y configuración del cliente

AWS Bedrock expone dos clientes principales en `boto3`:
- **`bedrock`**: para operaciones de gestión (listar modelos, consultar info)
- **`bedrock-runtime`**: para invocar modelos y obtener respuestas

In [ ]:
import boto3
import json

# Región donde tienes Bedrock habilitado
REGION = AWS_DEFAULT_REGION

# Cliente para gestión (listar modelos, etc.)
bedrock_client = boto3.client(
    service_name="bedrock",
    region_name=REGION
)

# Cliente para invocar modelos
bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name=REGION
)

print("✅ Clientes de Bedrock creados correctamente")

✅ Clientes de Bedrock creados correctamente


## 3. Listar modelos disponibles

Puedes consultar qué modelos están disponibles en tu cuenta y región usando `list_foundation_models`.

In [ ]:
# Listar todos los modelos base disponibles
response = bedrock_client.list_foundation_models()
modelos = response["modelSummaries"]

print(f"Total de modelos disponibles: {len(modelos)}\n")
print(f"{'Proveedor':<15} {'Model ID':<50} {'Modalidades'}")
print("-" * 90)

for modelo in modelos:
    provider = modelo.get("providerName", "N/A")
    model_id = modelo.get("modelId", "N/A")
    modalities = ", ".join(modelo.get("inputModalities", []))
    print(f"{provider:<15} {model_id:<50} {modalities}")

Total de modelos disponibles: 60

Proveedor       Model ID                                           Modalidades
------------------------------------------------------------------------------------------
Google          google.gemma-3-4b-it                               TEXT, IMAGE
NVIDIA          nvidia.nemotron-nano-12b-v2                        TEXT, IMAGE
Anthropic       anthropic.claude-sonnet-4-20250514-v1:0            TEXT, IMAGE
Anthropic       anthropic.claude-opus-4-7                          TEXT, IMAGE
Anthropic       anthropic.claude-haiku-4-5-20251001-v1:0           TEXT, IMAGE
OpenAI          openai.gpt-oss-safeguard-120b                      TEXT
Google          google.gemma-3-27b-it                              TEXT, IMAGE
OpenAI          openai.gpt-oss-120b-1:0                            TEXT
TwelveLabs      twelvelabs.marengo-embed-3-0-v1:0                  TEXT, IMAGE, SPEECH, VIDEO
Anthropic       anthropic.claude-sonnet-4-5-20250929-v1:0          TEXT, IMAGE
Twelv

In [ ]:
# Filtrar solo modelos de texto de Anthropic
modelos_anthropic = [
    m for m in modelos
    if m.get("providerName") == "Anthropic"
    and "TEXT" in m.get("inputModalities", [])
]

print("Modelos de Anthropic disponibles:")
for m in modelos_anthropic:
    print(f"  - {m['modelId']}")

Modelos de Anthropic disponibles:
  - anthropic.claude-sonnet-4-20250514-v1:0
  - anthropic.claude-opus-4-7
  - anthropic.claude-haiku-4-5-20251001-v1:0
  - anthropic.claude-sonnet-4-5-20250929-v1:0
  - anthropic.claude-sonnet-4-6
  - anthropic.claude-opus-4-5-20251101-v1:0
  - anthropic.claude-opus-4-6-v1
  - anthropic.claude-3-sonnet-20240229-v1:0:28k
  - anthropic.claude-3-sonnet-20240229-v1:0:200k
  - anthropic.claude-3-sonnet-20240229-v1:0
  - anthropic.claude-3-haiku-20240307-v1:0:48k
  - anthropic.claude-3-haiku-20240307-v1:0
  - anthropic.claude-3-7-sonnet-20250219-v1:0


## 4. Tu primera invocación: Hola Bedrock

Vamos a invocar Claude usando la API **Messages** de Anthropic.

La estructura del body varía según el proveedor del modelo. Para Claude (Anthropic) se usa el formato Messages API.

In [ ]:
# ID del modelo a usar
MODEL_ID = "eu.anthropic.claude-haiku-4-5-20251001-v1:0"

# Construcción del body de la petición (formato Anthropic Messages API)
body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 512,
    "messages": [{"role": "user", "content": "Hello, Claude"}]
}

# Invocar el modelo
response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,
    body=json.dumps(body),
    contentType="application/json",
    accept="application/json"
)

# Decodificar la respuesta
response_body = json.loads(response["body"].read())

print("=== RESPUESTA DEL MODELO ===")
print(response_body["content"][0]["text"])

=== RESPUESTA DEL MODELO ===
Hello! It's nice to meet you. How can I help you today?


## 5. Explorar la estructura completa de la respuesta

La respuesta contiene metadatos útiles además del texto generado.

In [ ]:
print("Estructura completa de la respuesta:")
print(json.dumps(response_body, indent=2, ensure_ascii=False))

Estructura completa de la respuesta:
{
  "model": "claude-haiku-4-5-20251001",
  "id": "msg_bdrk_011nZ1aAFgtHJwmD1df7vZeE",
  "type": "message",
  "role": "assistant",
  "content": [
    {
      "type": "text",
      "text": "Hello! It's nice to meet you. How can I help you today?"
    }
  ],
  "stop_reason": "end_turn",
  "stop_sequence": null,
  "usage": {
    "input_tokens": 10,
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "cache_creation": {
      "ephemeral_5m_input_tokens": 0,
      "ephemeral_1h_input_tokens": 0
    },
    "output_tokens": 19
  }
}


In [ ]:
# Extraer información relevante
texto_respuesta = response_body["content"][0]["text"]
tokens_entrada   = response_body["usage"]["input_tokens"]
tokens_salida    = response_body["usage"]["output_tokens"]
stop_reason      = response_body["stop_reason"]

print(f"📝 Texto generado:\n{texto_respuesta}")
print(f"\n📊 Tokens de entrada  : {tokens_entrada}")
print(f"📊 Tokens de salida   : {tokens_salida}")
print(f"🛑 Motivo de parada   : {stop_reason}")

📝 Texto generado:
Hello! It's nice to meet you. How can I help you today?

📊 Tokens de entrada  : 10
📊 Tokens de salida   : 19
🛑 Motivo de parada   : end_turn


## 6. Ajustar parámetros de inferencia

Puedes controlar el comportamiento del modelo con estos parámetros:

| Parámetro | Descripción | Rango |
|-----------|-------------|-------|
| `max_tokens` | Máximo de tokens a generar | 1 - 4096 |
| `temperature` | Aleatoriedad de la respuesta | 0.0 - 1.0 |
| `top_p` | Nucleus sampling | 0.0 - 1.0 |
| `stop_sequences` | Secuencias que detienen la generación | Lista de strings |

In [ ]:
def invocar_claude(prompt, temperature=0.7, max_tokens=512, model_id=MODEL_ID):
    """Función auxiliar para invocar Claude en Bedrock."""
    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "temperature": temperature,
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }
    response = bedrock_runtime.invoke_model(
        modelId=model_id,
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json"
    )
    resultado = json.loads(response["body"].read())
    return resultado["content"][0]["text"]


# Comparar temperatura baja vs alta
prompt_test = "Escribe una frase creativa sobre la inteligencia artificial."

print("🧊 Temperature = 0.0 (determinista):")
print(invocar_claude(prompt_test, temperature=0.0))

print("\n🔥 Temperature = 1.0 (más creativo):")
print(invocar_claude(prompt_test, temperature=1.0))

## 7. Conversaciones multi-turno

La API Messages soporta conversaciones con historial. Hay que pasar el historial completo en cada llamada.

In [ ]:
def chat_bedrock(historial, nuevo_mensaje, model_id=MODEL_ID):
    """Simula una conversación multi-turno con Bedrock."""
    historial.append({"role": "user", "content": nuevo_mensaje})

    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 512,
        "messages": historial
    }
    response = bedrock_runtime.invoke_model(
        modelId=model_id,
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json"
    )
    resultado = json.loads(response["body"].read())
    respuesta_texto = resultado["content"][0]["text"]

    historial.append({"role": "assistant", "content": respuesta_texto})
    return respuesta_texto, historial


# Iniciar conversación
historial = []

resp1, historial = chat_bedrock(historial, "Hola, me llamo Carlos y estoy aprendiendo Bedrock.")
print(f"Turno 1 - Asistente: {resp1}\n")

resp2, historial = chat_bedrock(historial, "¿Recuerdas cómo me llamo?")
print(f"Turno 2 - Asistente: {resp2}\n")

print(f"Total de mensajes en el historial: {len(historial)}")

## 8. Invocar Amazon Titan (otro proveedor)

Cada proveedor tiene su propio formato de body. Aquí vemos Amazon Titan Text.

In [ ]:
TITAN_MODEL_ID = "amazon.titan-text-express-v1"

# Formato específico de Amazon Titan
body_titan = {
    "inputText": "Explica en una oración qué es el machine learning.",
    "textGenerationConfig": {
        "maxTokenCount": 256,
        "temperature": 0.5,
        "topP": 0.9
    }
}

response_titan = bedrock_runtime.invoke_model(
    modelId=TITAN_MODEL_ID,
    body=json.dumps(body_titan),
    contentType="application/json",
    accept="application/json"
)

resultado_titan = json.loads(response_titan["body"].read())
print("Respuesta de Amazon Titan:")
print(resultado_titan["results"][0]["outputText"])

## 9. Resumen y próximos pasos

En este notebook aprendiste a:

✅ Crear clientes `bedrock` y `bedrock-runtime` con boto3  
✅ Listar y filtrar modelos disponibles  
✅ Invocar Claude con la Messages API  
✅ Interpretar la estructura de la respuesta (tokens, stop_reason)  
✅ Ajustar parámetros como `temperature` y `max_tokens`  
✅ Mantener conversaciones multi-turno  
✅ Invocar modelos de distintos proveedores (Anthropic vs Amazon)  

**Siguiente notebook →** Prompt Engineering aplicado a Bedrock